In [28]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
# Sélection du scope de données à traiter (sources, défauts, diamètres, positions, charges, canaux)
selection = data.selectionner(
    sources=("12kDE", "12kFE", "Normal"),
    defauts=("IR", "B", "OR"),
    diametres=(7, 14, 21), positions_or=("6",),
    charges=(0, 1, 2, 3),
    canaux={"12kDE": ("DE",), 
            "12kFE": ("FE",), 
            "Normal": ("DE","FE")},
)

In [38]:
# Importation des données sélectionnées et ajout des indicateurs temporels et spectraux (ajout de ceux de Mourad)
table = features.construire_table_SpecMourad(selection, data.charger)

In [55]:
# Preparation pour le modeling + modeling
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
i=0
for contexte in ["DE→DE", "DE→FE"]:
    for target in ["defaut", "classe"]:
        i += 1

        COLONNES_INDICATEURS = indicateurs.COLONNES_TEMPORELS

        if i == 1:
            print("\n#=====================================#")
            print("#====== LogReg VS Random Forest ======#")
            print("#=====================================#")
            print("\n# Features #")
            display(table[COLONNES_INDICATEURS].head(5))


        if contexte == "DE→DE":
            X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling(table, COLONNES_INDICATEURS, target=target)
        elif contexte == "DE→FE":
            X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_DE_FE(table, COLONNES_INDICATEURS, canal_train="DE", canal_test="FE", target=target)
        print("##############################")
        print(f"###### Contexte test {i} #######")
        print(f"###### Train→Test : {contexte}")
        print(f"###### target : {target}")

        X_train, y_train, X_test, y_test = features.equilibrer(X_train, y_train, X_test, y_test, type="train2test")
        if contexte == "DE→DE":
            display(pd.DataFrame({"train DE": y_train.value_counts(), "test DE": y_test.value_counts()}))
        elif contexte == "DE→FE":
            display(pd.DataFrame({"train DE": y_train.value_counts(), "test FE": y_test.value_counts()}))
        
        # print("\n#=====================================#")
        # print("#====== LogReg VS Random Forest ======#")
        # print("#=====================================#")

        modeles = {
            "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
        }
        tableau = models.entrainer_comparer(modeles, X_train, y_train, X_test, y_test, report=False, dispMatrice=False)
        tableau

        feats = {}
        for feature, importance in zip(X_test.columns, modeles["Random Forest"].feature_importances_):
            feats[feature] = importance
        importances = pd.DataFrame.from_dict(feats, orient='index')
        importances = importances.rename(columns={0:'Importance'})
        importances = importances.sort_values('Importance', ascending=False)
        display(importances)

        print("\n")


#=====================================#
#====== LogReg VS Random Forest ======#
#=====================================#

# Features #


,RMS,Pic,Crete,Kurtosis
0,0.073348,0.272869,3.720207,2.859637
1,0.078434,0.226765,2.891156,2.863790
2,0.073608,0.249295,3.386784,2.794364
3,0.073188,0.247418,3.380577,2.935528
4,0.071653,0.224679,3.135669,2.833588


##############################
###### Contexte test 1 #######
###### Train→Test : DE→DE
###### target : defaut


,train DE,test DE
defaut,,
B,702,300
IR,702,300
OR,702,201
Sain,702,404


=== LogReg ===
Accuracy          : 0.782
Balanced accuracy : 0.740
F1-macro          : 0.743


/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detect

... (20 outputs left out)

/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detect

=== Random Forest ===
Accuracy          : 0.221
Balanced accuracy : 0.188
F1-macro          : 0.166


/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detection/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/robin/BDS_Fault_Detect

,Importance
RMS,0.409040
Pic,0.237633
Kurtosis,0.235230
Crete,0.118097
